# 🔄 XFarmaCare — Scoring Diario de Churn
## Pipeline Recurrente (Background Job)

**Propósito:** Este notebook se ejecuta diariamente de forma automática.
Carga el modelo entrenado, procesa los datos más recientes y actualiza
las probabilidades de churn de toda la cartera.

**Modo de uso:**
```bash
# Ejecutar como script en segundo plano
jupyter nbconvert --to script 02_modelo_churn_scoring_diario.ipynb
nohup python 02_modelo_churn_scoring_diario.py &
```

O programar con cron:
```bash
0 6 * * * cd /ruta/proyecto && python 02_modelo_churn_scoring_diario.py >> logs/scoring.log 2>&1
```
---

In [ ]:
# === Setup ===
import pandas as pd
import numpy as np
import joblib
import os
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Rutas
DATA_DIR = "data/"
MODEL_DIR = "models/"
OUTPUT_DIR = "outputs/"
LOG_FILE = "outputs/scoring_log.csv"

print(f"[{datetime.now()}] Iniciando scoring diario de churn...")

In [ ]:
# === Cargar modelo y componentes ===
modelo = joblib.load(f"{MODEL_DIR}modelo_churn_xfarmacare.pkl")
scaler = joblib.load(f"{MODEL_DIR}scaler_churn.pkl")
le_dict = joblib.load(f"{MODEL_DIR}label_encoders_churn.pkl")
feature_cols = joblib.load(f"{MODEL_DIR}feature_cols_churn.pkl")

print(f"Modelo cargado: {type(modelo).__name__}")
print(f"Features esperadas: {len(feature_cols)}")

In [ ]:
# === Cargar datos frescos ===
# En producción, esto leería de la base de datos en tiempo real.
# Aquí simula leyendo los CSVs actualizados.

clientes = pd.read_csv(f"{DATA_DIR}dim_clientes.csv")
transacciones = pd.read_csv(f"{DATA_DIR}fact_transacciones.csv", parse_dates=["fecha_compra"])
callcenter = pd.read_csv(f"{DATA_DIR}fact_interacciones_callcenter.csv", parse_dates=["fecha_interaccion"])
adherencia = pd.read_csv(f"{DATA_DIR}fact_adherencia_terapeutica.csv")

HOY = pd.Timestamp.now().normalize()
print(f"Fecha de scoring: {HOY.date()}")
print(f"Clientes a procesar: {len(clientes):,}")

In [ ]:
# === Reconstruir features (misma lógica que el entrenamiento) ===
# Esto garantiza que el scoring use exactamente las mismas transformaciones.

def construir_features(clientes_df, txn_df, cc_df, adh_df, fecha_referencia):
    """
    Construye el dataset analítico a nivel cliente.
    Replica exactamente la lógica del notebook de entrenamiento.
    """
    # --- Features de compra ---
    feat_compras = txn_df.groupby('customer_id').agg(
        total_transacciones=('transaction_id', 'count'),
        total_gastado_dop=('total_venta_dop', 'sum'),
        promedio_ticket_dop=('total_venta_dop', 'mean'),
        max_ticket_dop=('total_venta_dop', 'max'),
        productos_distintos=('product_id', 'nunique'),
        canales_usados=('canal_compra', 'nunique'),
        primera_compra=('fecha_compra', 'min'),
        ultima_compra=('fecha_compra', 'max'),
        descuento_promedio=('descuento_porcentaje', 'mean'),
        compras_receta_recurrente=('fue_receta_recurrente', 'sum')
    ).reset_index()
    
    feat_compras['recencia_dias'] = (fecha_referencia - feat_compras['ultima_compra']).dt.days
    feat_compras['antiguedad_dias'] = (fecha_referencia - feat_compras['primera_compra']).dt.days
    feat_compras['frecuencia_mensual'] = (
        feat_compras['total_transacciones'] / np.maximum(feat_compras['antiguedad_dias'] / 30, 1)
    ).round(2)
    
    canal_dom = txn_df.groupby('customer_id')['canal_compra'].agg(
        lambda x: x.value_counts().index[0]).reset_index()
    canal_dom.columns = ['customer_id', 'canal_dominante']
    feat_compras = feat_compras.merge(canal_dom, on='customer_id', how='left')
    
    # Tendencia de gasto
    fecha_3m = fecha_referencia - pd.Timedelta(days=90)
    fecha_6m = fecha_referencia - pd.Timedelta(days=180)
    g_rec = txn_df[txn_df['fecha_compra'] >= fecha_3m].groupby('customer_id')['total_venta_dop'].sum()
    g_ant = txn_df[(txn_df['fecha_compra'] >= fecha_6m) & (txn_df['fecha_compra'] < fecha_3m)].groupby('customer_id')['total_venta_dop'].sum()
    tend = pd.DataFrame({'gasto_reciente': g_rec, 'gasto_anterior': g_ant}).fillna(0)
    tend['tendencia_gasto'] = np.where(tend['gasto_anterior'] > 0,
        ((tend['gasto_reciente'] - tend['gasto_anterior']) / tend['gasto_anterior']).clip(-1, 5), 0)
    feat_compras = feat_compras.merge(tend[['tendencia_gasto']], left_on='customer_id', right_index=True, how='left')
    feat_compras['tendencia_gasto'] = feat_compras['tendencia_gasto'].fillna(0)
    feat_compras = feat_compras.drop(columns=['primera_compra', 'ultima_compra'])
    
    # --- Features de sentimiento ---
    feat_sent = cc_df.groupby('customer_id').agg(
        total_interacciones=('interaction_id', 'count'),
        interacciones_negativas=('sentimiento', lambda x: (x == 'NEGATIVO').sum()),
        interacciones_positivas=('sentimiento', lambda x: (x == 'POSITIVO').sum()),
        puntaje_neg_promedio=('puntaje_negativo', 'mean'),
        puntaje_pos_promedio=('puntaje_positivo', 'mean'),
        estrellas_promedio=('estrellas', 'mean'),
        tasa_resolucion=('resuelto', 'mean'),
        tiempo_espera_promedio=('tiempo_espera_seg', 'mean'),
        duracion_promedio_min=('duracion_minutos', 'mean')
    ).reset_index()
    feat_sent['ratio_negatividad'] = (
        feat_sent['interacciones_negativas'] / feat_sent['total_interacciones']).round(3)
    
    motivo_f = cc_df.groupby('customer_id')['motivo'].agg(
        lambda x: x.value_counts().index[0]).reset_index()
    motivo_f.columns = ['customer_id', 'motivo_mas_frecuente']
    feat_sent = feat_sent.merge(motivo_f, on='customer_id', how='left')
    
    # --- Features de adherencia ---
    feat_adh = adh_df.groupby('customer_id').agg(
        tasa_adherencia_promedio=('tasa_adherencia_periodo', 'mean'),
        tasa_adherencia_min=('tasa_adherencia_periodo', 'min'),
        dias_gap_promedio=('dias_gap', 'mean'),
        dias_gap_max=('dias_gap', 'max'),
        total_ciclos_tratamiento=('adherencia_id', 'count'),
        ciclos_con_retraso=('dias_gap', lambda x: (x > 7).sum())
    ).reset_index()
    feat_adh['ratio_retraso'] = (
        feat_adh['ciclos_con_retraso'] / feat_adh['total_ciclos_tratamiento']).round(3)
    
    def adh_trend(group):
        if len(group) < 4: return 0
        return round(group['tasa_adherencia_periodo'].tail(3).mean() - 
                     group['tasa_adherencia_periodo'].iloc[:-3].mean(), 3)
    trend_a = adh_df.groupby('customer_id').apply(adh_trend).reset_index()
    trend_a.columns = ['customer_id', 'tendencia_adherencia']
    feat_adh = feat_adh.merge(trend_a, on='customer_id', how='left')
    
    # --- Unir todo ---
    df = clientes_df[['customer_id', 'edad', 'sexo', 'provincia', 'zona',
                       'tipo_seguro', 'es_cronico', 'condicion_cronica',
                       'programa_lealtad', 'nivel_lealtad', 'canal_preferido',
                       'ingreso_estimado']].copy()
    
    df = df.merge(feat_compras, on='customer_id', how='left')
    df = df.merge(feat_sent, on='customer_id', how='left')
    df = df.merge(feat_adh, on='customer_id', how='left')
    df = df.fillna(0)
    df['motivo_mas_frecuente'] = df['motivo_mas_frecuente'].replace(0, 'Sin contacto')
    
    return df

# Ejecutar
df_scoring = construir_features(clientes, transacciones, callcenter, adherencia, HOY)
print(f"Dataset para scoring: {df_scoring.shape}")

In [ ]:
# === Codificar categóricas con los mismos encoders ===
cols_categoricas = ['sexo', 'zona', 'tipo_seguro', 'condicion_cronica',
                    'nivel_lealtad', 'canal_preferido', 'ingreso_estimado',
                    'canal_dominante', 'motivo_mas_frecuente']

for col in cols_categoricas:
    if col in df_scoring.columns and col in le_dict:
        df_scoring[col] = df_scoring[col].fillna('Desconocido').astype(str)
        # Manejar valores nuevos que no vio el encoder
        known = set(le_dict[col].classes_)
        df_scoring[col] = df_scoring[col].apply(lambda x: x if x in known else 'Desconocido')
        if 'Desconocido' not in known:
            le_dict[col].classes_ = np.append(le_dict[col].classes_, 'Desconocido')
        df_scoring[f'{col}_enc'] = le_dict[col].transform(df_scoring[col])

# Preparar features en el mismo orden que el entrenamiento
X_score = df_scoring[feature_cols].fillna(0)

# Aplicar scaler si el modelo lo necesita
modelo_nombre = type(modelo).__name__
if 'Logistic' in modelo_nombre:
    X_score_proc = scaler.transform(X_score)
else:
    X_score_proc = X_score

print(f"Features listas para scoring: {X_score.shape}")

In [ ]:
# === Generar predicciones ===
probabilidades = modelo.predict_proba(X_score_proc)[:, 1]

# Crear output
output = pd.DataFrame({
    'customer_id': df_scoring['customer_id'],
    'probabilidad_churn': probabilidades.round(4),
    'riesgo_churn': pd.cut(probabilidades,
        bins=[0, 0.20, 0.40, 0.60, 0.80, 1.0],
        labels=['Muy Bajo', 'Bajo', 'Medio', 'Alto', 'Crítico']),
    'fecha_scoring': HOY.strftime('%Y-%m-%d'),
    'modelo_version': 'v1.0_GBM'
})

# Guardar (sobreescribe el archivo diario)
output.to_csv(f"{OUTPUT_DIR}output_churn_scores.csv", index=False)

# Log de ejecución
log_entry = pd.DataFrame([{
    'fecha_ejecucion': datetime.now().isoformat(),
    'clientes_procesados': len(output),
    'churn_critico': (output['riesgo_churn'] == 'Crítico').sum(),
    'churn_alto': (output['riesgo_churn'] == 'Alto').sum(),
    'prob_media': probabilidades.mean().round(4)
}])

if os.path.exists(LOG_FILE):
    log_entry.to_csv(LOG_FILE, mode='a', header=False, index=False)
else:
    log_entry.to_csv(LOG_FILE, index=False)

print(f"\n{'='*50}")
print(f"SCORING COMPLETADO — {HOY.date()}")
print(f"{'='*50}")
print(f"Clientes procesados: {len(output):,}")
print(f"\nDistribución de riesgo:")
print(output['riesgo_churn'].value_counts().sort_index().to_string())
print(f"\nClientes en riesgo CRÍTICO: {(output['riesgo_churn']=='Crítico').sum()}")
print(f"Clientes en riesgo ALTO: {(output['riesgo_churn']=='Alto').sum()}")

## ✅ Scoring Diario Completado
- Resultados en `outputs/output_churn_scores.csv`
- Log de ejecución en `outputs/scoring_log.csv`
- Conectado por `customer_id` al resto del modelo estrella